In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.pipeline import Pipeline
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [13]:
df = pd.read_excel("/content/drive/MyDrive/MakineOgrenmesiYL/ML/Dry_Bean_Dataset.xlsx")


In [14]:
#Kod çalışırken oluşabilecek uyarılar gizlenmiştir.
warnings.filterwarnings('ignore')

# Rastgele işlemler için başlangıç değeri belirlenir, böylece her çalıştırmada aynı sonuç alınır
np.random.seed(42)



# Temel bilgileri göster
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nClass distribution:")
print(df['Class'].value_counts())



##  Bölüm 1: Veri Ön İşleme (Preprocessing)

# 2. Eksik Değerler Ekleme
# İki rastgele sütuna %5 oranında eksik değer ekle, Bir sütuna %35 oranında eksik değer ekle.
for col in ['MajorAxisLength', 'ShapeFactor1']:
    df.loc[df.sample(frac=0.05).index, col] = np.nan

df.loc[df.sample(frac=0.35).index, 'roundness'] = np.nan

# 2a. Eksik değerleri kontrol et
print("\nMissing values after addition:")  # Ekleme sonrası eksik değerler
print(df.isnull().sum())

# 2b. %5 eksik değeri medyan ile doldurulur
for col in ['MajorAxisLength', 'ShapeFactor1']:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

# 2c. %35 eksik değere sahip sütun, çok fazla eksik olduğu için silinir
df.drop('roundness', axis=1, inplace=True)

# 3. Aykırı Değer Tespiti ve İşlenmesi (IQR Yöntemi)
def treat_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

   # Aykırı değerleri sınırla (cap)
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
    return df

# Tüm sayısal sütunlara uygula ('Class' hedef değişkenini hariç tut)
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
# Hedef değişken yanlışlıkla dahil edildiyse dahil etme
numerical_cols = numerical_cols.drop('Class') if 'Class' in numerical_cols else numerical_cols

for col in numerical_cols:
    df = treat_outliers(df, col)

#4. Özellik Ölçekleme (Feature Scaling)
scaler = StandardScaler()
X = df.drop('Class', axis=1)
y = df['Class']
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 5. Kategorik Verilerin Kodlanması ('Class' sütunu)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

## ## Bölüm 2: Özellik Seçimi ve Boyut Azaltma

# 6. PCA (Temel Bileşen Analizi)
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Açıklanan varyans ortalamadan büyük olduğu bileşen sayısını belirle
explained_variance = pca.explained_variance_ratio_
average_variance = explained_variance.mean()
n_components_pca = sum(explained_variance > average_variance)

print(f"\nPCA: Selected {n_components_pca} components where explained variance > average ({average_variance:.3f})")

# Açıklanan varyansı görselleştir
plt.figure(figsize=(10, 6))
plt.plot(np.cumsum(explained_variance))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Explained Variance')
plt.axhline(y=0.95, color='r', linestyle='--')
plt.axvline(x=n_components_pca, color='g', linestyle='--')
plt.grid()
plt.savefig('pca_explained_variance.png')
plt.show()

# Seçilen bileşen sayısıyla PCA uygula
pca = PCA(n_components=n_components_pca)
X_pca = pca.fit_transform(X_scaled)

# İlk iki PCA bileşenini görselleştir
plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette='viridis')
plt.title('First Two PCA Components')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.savefig('pca_components.png')
plt.show()

# 7. LDA (Doğrusal Ayrıştırma Analizi)
lda = LDA(n_components=3)
X_lda = lda.fit_transform(X_scaled, y_encoded)

# İlk iki LDA bileşenini görselleştir
plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_lda[:, 0], y=X_lda[:, 1], hue=y, palette='viridis')
plt.title('First Two LDA Components')
plt.xlabel('LDA Component 1')
plt.ylabel('LDA Component 2')
plt.savefig('lda_components.png')
plt.show()

## Bölüm 3: Modelleme ve Değerlendirme (Düzenlenmiş Sürüm)

# Veri temsillerini hazırla – tutarlı indeksleme için numpy array’lerine dönüştür
data_representations = {
    'Original': X_scaled.values,
    'PCA': X_pca,
    'LDA': X_lda
}

# Sınıflandırıcıları tanımla
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    'Naive Bayes': GaussianNB()
}

# Grid search için hiperparametreleri tanımla
param_grids = {
    'Logistic Regression': {'C': [0.1, 1, 10]},
    'Decision Tree': {'max_depth': [3, 5, None]},
    'Random Forest': {'n_estimators': [50, 100], 'max_depth': [3, 5]},
    'XGBoost': {'n_estimators': [50, 100], 'max_depth': [3, 5]},
    'Naive Bayes': {}
}

# Sonuçları depolamak için alan hazırla
results = []

# İç içe çapraz doğrulama: 5 dış kat, 3 iç kat
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Her bir veri temsili için:
for rep_name, X_rep in data_representations.items():
    print(f"\n=== Evaluating on {rep_name} representation ===")

     # Her bir sınıflandırıcı için:
    for clf_name, clf in classifiers.items():
        print(f"\n-- {clf_name} --")

       # Bu sınıflandırıcıya özel metrik depolama alanını başlat
        accuracies = []
        precisions = []
        recalls = []
        f1_scores = []
        roc_aucs = []

        # ROC eğrisi için – en iyi dış katman sonuçlarını sakla
        best_outer_fold = None
        best_roc_auc = 0
        best_y_test = None
        best_y_probs = None

        # Outer CV loop  ---# Dış çapraz doğrulama döngüsü
        for i, (train_idx, test_idx) in enumerate(outer_cv.split(X_rep, y_encoded)):
            X_train, X_test = X_rep[train_idx], X_rep[test_idx]
            y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

            # Inner CV for hyperparameter tuning  --# Hiperparametre ayarı için iç çapraz doğrulama
            if param_grids[clf_name]:   # Ayarlanacak parametre yoksa atla
                grid_search = GridSearchCV(clf, param_grids[clf_name], cv=inner_cv, scoring='accuracy')
                grid_search.fit(X_train, y_train)
                best_clf = grid_search.best_estimator_
            else:
                best_clf = clf
                best_clf.fit(X_train, y_train)

            # Test seti üzerinde değerlendir
            y_pred = best_clf.predict(X_test)

            # ROC eğrisi için predict_proba gerekir, bazı sınıflandırıcılarda olmayabilir
            try:
                y_probs = best_clf.predict_proba(X_test)
                roc_auc = roc_auc_score(y_test, y_probs, multi_class='ovr')
                roc_aucs.append(roc_auc)

               # ROC eğrisi için en iyi dış katmanı takip et
                if roc_auc > best_roc_auc:
                    best_roc_auc = roc_auc
                    best_outer_fold = i
                    best_y_test = y_test
                    best_y_probs = y_probs
            except AttributeError:
                y_probs = None
                roc_auc = None

           # Diğer metrikleri hesapla
            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, average='weighted')
            rec = recall_score(y_test, y_pred, average='weighted')
            f1 = f1_score(y_test, y_pred, average='weighted')

            accuracies.append(acc)
            precisions.append(prec)
            recalls.append(rec)
            f1_scores.append(f1)

        # Metriklerin ortalamasını ve standart sapmasını hesapla
        mean_acc = np.mean(accuracies)
        std_acc = np.std(accuracies)
        mean_prec = np.mean(precisions)
        std_prec = np.std(precisions)
        mean_rec = np.mean(recalls)
        std_rec = np.std(recalls)
        mean_f1 = np.mean(f1_scores)
        std_f1 = np.std(f1_scores)
        mean_roc_auc = np.mean(roc_aucs) if roc_aucs else np.nan
        std_roc_auc = np.std(roc_aucs) if roc_aucs else np.nan

        # Sonuçları depola
        results.append({
            'Data Representation': rep_name,
            'Classifier': clf_name,
            'Accuracy': f"{mean_acc:.3f} ± {std_acc:.3f}",
            'Precision': f"{mean_prec:.3f} ± {std_prec:.3f}",
            'Recall': f"{mean_rec:.3f} ± {std_rec:.3f}",
            'F1 Score': f"{mean_f1:.3f} ± {std_f1:.3f}",
            'ROC AUC': f"{mean_roc_auc:.3f} ± {std_roc_auc:.3f}" if not np.isnan(mean_roc_auc) else "N/A"
        })

        # Eğer olasılıklar varsa en iyi dış katman için ROC eğrisini çiz
        if best_y_probs is not None:
            plt.figure(figsize=(10, 8))
            n_classes = len(label_encoder.classes_)

            # Her sınıf için ROC eğrisi ve alanını hesapla
            fpr = dict()
            tpr = dict()
            roc_auc = dict()

            for i in range(n_classes):
                fpr[i], tpr[i], _ = roc_curve((best_y_test == i).astype(int), best_y_probs[:, i])
                roc_auc[i] = roc_auc_score((best_y_test == i).astype(int), best_y_probs[:, i])

           # Tüm ROC eğrilerini çiz
            colors = plt.cm.rainbow(np.linspace(0, 1, n_classes))
            for i, color in zip(range(n_classes), colors):
                plt.plot(fpr[i], tpr[i], color=color, lw=2,
                         label='ROC curve of class {0} (AUC = {1:0.2f})'
                         ''.format(label_encoder.classes_[i], roc_auc[i]))

            plt.plot([0, 1], [0, 1], 'k--', lw=2)
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title(f'ROC Curves for {clf_name} on {rep_name} (Fold {best_outer_fold+1})')
            plt.legend(loc="lower right")
            plt.savefig(f'roc_{rep_name}_{clf_name}.png')
            plt.show()

# Sonuçları DataFrame’e dönüştür
results_df = pd.DataFrame(results)
print("\nPerformance Metrics Summary:")
print(results_df.to_markdown(index=False))

# En iyi model kombinasyonunu bul
# Gerekirse ROC AUC’si olmayan (N/A) sonuçları filtrele
valid_results = [r for r in results if r['ROC AUC'] != "N/A"]
best_result = max(valid_results, key=lambda x: float(x['Accuracy'].split()[0]))
print(f"\nBest model: {best_result['Classifier']} on {best_result['Data Representation']} data")
print(f"Accuracy: {best_result['Accuracy']}")
print(f"ROC AUC: {best_result['ROC AUC']}")

Output hidden; open in https://colab.research.google.com to view.